# Figure 5. Precision–recall and ROC comparisons against in-sample PIPs

Compare variant PIPs from SuSiE-RSS and SuSiE-rRSS with the in-sample PIPs. Read the fits saved by the colocalization jobs (`res_<gene or region>.rds`), with one list element per tissue or trait. All fits use the same harmonized variants (`variant_ids`).

| `dataset` | fits (`res_*.rds`) and coloc records (`coloc/result/`) | reference LD | in-sample comparison |
|---|---|---|---|
| `GTEx` (Fig 5A, `16`) | `Gtex_colocalization/` | ADSP sketch | `susie_res`: individual-level SuSiE |
| `UKB` (Fig 5B, `18`) | `UKB_coloclization/` | ADSP sketch | `insample_res`: SuSiE-RSS with in-sample LD |
| `GTEx_1000G` (Fig S11A, `17`) | `Gtex_colocalization_1000G/` | 1000 Genomes EUR | `susie_res` |
| `UKB_1000G` (Fig S11B, `19`) | `UKB_coloclization_1000_genome/` | 1000 Genomes EUR | `insample_res` |

GTEx fits are in `res/`, UKB fits in `result/`. Use the same `insample` fit as the colocalization step `[coloc]`.
- **SuSiE-RSS:** `susie_rss_res`.
- **SuSiE-rRSS:** Use `susie_eb_mix_res` when `R_reliability_flag` is FALSE. Otherwise use the SER model `susie_ser_res` (the `eb+SER` rule of `add_ser_hybrids()` in `R/utils.R`, which Fig 5 uses).

Two steps:
1. `[pr_units]` (interactive, minutes): from the coloc records, every unit (gene × tissue, or region × trait) with a record for SuSiE-RSS, SuSiE-rRSS (eb_mix) or the SER model. A record is written only when the in-sample fit or that fit has a credible set, so this pool contains every unit where the in-sample fit, SuSiE-RSS or SuSiE-rRSS has a credible set (plus a few where only an unused SER model has one; these are dropped in step 2). Sample `n_units × oversample` units using `seed`, record their draw order in `rank`, and group them into chunks without splitting a `res_*.rds` file.
2. `[pr_extract]` (one task per chunk): reads each file once and, per unit, saves the three PIP vectors and the credible-set counts. `eligible` = the in-sample fit, SuSiE-RSS or SuSiE-rRSS (after fallback) has at least one credible set. Every variant is kept (`min_pip = 0`; about 2000 × 6000 rows per dataset). With `min_pip` > 0 only variants with SuSiE-RSS or SuSiE-rRSS PIP ≥ `min_pip` are kept and the others enter the curves only through the per-unit totals (`p`, Σ in-sample PIP, number of in-sample PIP ≥ 0.5), so the curves stay exact for thresholds ≥ `min_pip`.

`Data_Processing/Figure_5/Fig5_Power_FDR` keeps the first 2000 eligible units by `rank` per dataset.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fig5_PR_ROC/`: `units/<dataset>_pr_units.tsv`, `result/<dataset>_pr_chunk_<k>.rds` (`unit`: one row per unit; `var`: one row per kept variant).

## 1. Units

In [ ]:
[pr_units]
# Pool of units from the coloc records; a random draw in random order, in chunks of whole res files.
parameter: dataset = "GTEx"
# GTEx, UKB, GTEx_1000G or UKB_1000G
parameter: coloc_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/coloc/result")
parameter: n_units = 2000
# units drawn = n_units x oversample, so that n_units eligible ones remain after step 2
parameter: oversample = 1.3
parameter: n_chunks = 40
parameter: seed = 2026
parameter: cwd = path("output")
parameter: container = ""
output: f'{cwd:a}/{dataset}_pr_units.tsv'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    suppressPackageStartupMessages(library(tidyverse))
    files <- list.files(${coloc_dir:r}, pattern = "^coloc_.*\\.rds$", full.names = TRUE)
    recs  <- map_dfr(files, function(f) {
      x <- tryCatch(readRDS(f), error = function(e) NULL)
      if (!is.data.frame(x) || !nrow(x)) return(NULL)
      x %>% transmute(gene = as.character(gene), tissue = as.character(tissue), method)
    })
    cat(sprintf("%s: %d coloc files, %d records\n", "${dataset}", length(files), nrow(recs)))

    ## a record exists only when the in-sample fit or the tested fit has a credible set
    pool <- recs %>% filter(method %in% c("susie_rss_res", "susie_eb_mix_res", "susie_ser_res")) %>%
      distinct(gene, tissue) %>% arrange(gene, tissue)
    n_take <- min(nrow(pool), ceiling(${n_units} * ${oversample}))
    set.seed(${seed})
    units <- pool %>% slice_sample(n = n_take) %>% mutate(rank = row_number())
    ## chunks hold whole res files (one per gene / region), so each file is read once
    chunk_of <- units %>% distinct(gene) %>% mutate(chunk = (row_number() - 1L) %% ${n_chunks} + 1L)
    units <- units %>% left_join(chunk_of, by = "gene")
    write_tsv(units, ${_output:r})
    cat(sprintf("pool: %d units (%d genes / regions); drawn: %d units, %d files, %d chunks\n",
                nrow(pool), n_distinct(pool$gene), nrow(units), n_distinct(units$gene), n_distinct(units$chunk)))

In [ ]:
# run interactively on a login node (a few minutes: reads every coloc record)
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
nb="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/35.Fig5_PR_ROC_Extract.ipynb"
units_dir="${base_dir}/Fig5_PR_ROC/units"
mkdir -p ${units_dir}
## ADSP sketch (Fig 5A / 5B)
sos run "$nb" pr_units --dataset GTEx --coloc_dir ${base_dir}/Gtex_colocalization/coloc/result --cwd ${units_dir}
sos run "$nb" pr_units --dataset UKB  --coloc_dir ${base_dir}/UKB_coloclization/coloc/result  --cwd ${units_dir}
## 1000 Genomes (Fig S11A / S11B)
sos run "$nb" pr_units --dataset GTEx_1000G --coloc_dir ${base_dir}/Gtex_colocalization_1000G/coloc/result     --cwd ${units_dir}
sos run "$nb" pr_units --dataset UKB_1000G  --coloc_dir ${base_dir}/UKB_coloclization_1000_genome/coloc/result --cwd ${units_dir}
wc -l ${units_dir}/*_pr_units.tsv

## 2. Extract PIPs

In [ ]:
[pr_extract]
# One task per chunk: the in-sample, SuSiE-RSS and SuSiE-rRSS (after SER fallback) PIPs of each unit.
parameter: dataset = "GTEx"
parameter: res_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/res")
parameter: units_file = path("output/GTEx_pr_units.tsv")
# as in [pr_units]
parameter: n_chunks = 40
# variants kept: SuSiE-RSS or SuSiE-rRSS PIP >= min_pip (0 = all; curves exact for thresholds >= min_pip)
parameter: min_pip = 0.0
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "12h"
parameter: mem = "40G"
parameter: numThreads = 1
parameter: container = ""
chunk_ids = [str(i) for i in range(1, n_chunks + 1)]
input: for_each = "chunk_ids"
output: f'{cwd:a}/{dataset}_pr_chunk_{_chunk_ids}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{dataset}_{_chunk_ids}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    suppressPackageStartupMessages(library(tidyverse))
    dataset  <- "${dataset}"
    min_pip  <- ${min_pip}
    truth_nm <- if (startsWith(dataset, "GTEx")) "susie_res" else "insample_res"   # the in-sample fit of [coloc] in 16-19
    units <- read_tsv(${units_file:r}, col_types = cols(gene = "c", tissue = "c", rank = "i", chunk = "i")) %>%
      filter(chunk == ${_chunk_ids})
    n_cs <- function(fit) if (is.null(fit)) NA_integer_ else length(fit$sets$cs)

    one_unit <- function(r, gene, tissue, rank) {
      truth <- r[[truth_nm]]; rss <- r$susie_rss_res; eb <- r$susie_eb_mix_res; ser <- r$susie_ser_res
      flag <- eb$R_finite_diagnostics$R_reliability_flag
      flag <- if (is.null(flag) || !length(flag)) NA else as.logical(flag)[1]
      use_eb <- flag %in% FALSE                 # eb+SER rule (add_ser_hybrids): SER unless the flag is FALSE
      rrss <- if (use_eb) eb else ser
      p <- length(r$variant_ids)
      if (!p) p <- length(truth$pip)            # 17 (GTEx 1000G) saved variant_ids = colnames(harm$sketch) = NULL (its matrix is harm$geno)
      status <- if (is.null(r)) "no fits for this tissue / trait" else
                if (is.null(truth)) "in-sample fit missing" else
                if (is.null(rss)) "SuSiE-RSS fit missing" else
                if (is.null(eb)) "SuSiE-rRSS (eb_mix) fit missing" else
                if (is.null(rrss)) "SER fit missing" else
                if (!all(lengths(list(truth$pip, rss$pip, rrss$pip)) == p)) "PIP length mismatch" else "ok"
      unit <- tibble(rank = rank, gene = gene, tissue = tissue, status = status, p = p,
                     reliability_flag = flag, ser_fallback = !use_eb,
                     n_cs_truth = n_cs(truth), n_cs_rss = n_cs(rss), n_cs_rrss = n_cs(rrss),
                     eligible = NA, sum_pip_truth = NA_real_, n_pip05_truth = NA_integer_)
      if (status != "ok") return(list(unit = unit))
      pin <- as.numeric(truth$pip); prs <- as.numeric(rss$pip); prr <- as.numeric(rrss$pip)
      unit <- unit %>% mutate(eligible = n_cs_truth + n_cs_rss + n_cs_rrss > 0,
                              sum_pip_truth = sum(pin), n_pip05_truth = sum(pin >= 0.5))
      keep <- which(pmax(prs, prr) >= min_pip)
      list(unit = unit,
           var  = tibble(rank = rank, idx = keep, pip_truth = pin[keep], pip_rss = prs[keep], pip_rrss = prr[keep]))
    }

    out <- list()
    for (g in unique(units$gene)) {
      res <- tryCatch(readRDS(file.path(${res_dir:r}, paste0("res_", g, ".rds"))), error = function(e) NULL)
      ug  <- filter(units, gene == g)
      for (k in seq_len(nrow(ug)))
        out[[length(out) + 1]] <- one_unit(if (is.list(res)) res[[ug$tissue[k]]] else NULL, g, ug$tissue[k], ug$rank[k])
      rm(res); invisible(gc())
    }
    unit_tab <- map_dfr(out, "unit")
    var_tab  <- bind_rows(map(out, "var"))
    saveRDS(list(dataset = dataset, min_pip = min_pip, unit = unit_tab, var = var_tab), ${_output:r})
    print(count(unit_tab, status, eligible, ser_fallback))
    cat(sprintf("chunk %s: %d units, %d variants kept\n", "${_chunk_ids}", nrow(unit_tab), nrow(var_tab)))

Submit: one job per dataset (`[pr_units]` first).

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="Fig5_PR_ROC"
job_dir="${base_dir}/${job_name}"
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_DATASET
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=70000]
#BSUB -W 48:00
#BSUB -o WORK_DIR/log/JOB_NAME_DATASET.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_DATASET.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/35.Fig5_PR_ROC_Extract.ipynb" pr_extract \
    --dataset DATASET --res_dir RES_DIR --units_file WORK_DIR/units/DATASET_pr_units.tsv \
    --cwd WORK_DIR/result --mem 60G -J 4
EOF

for dataset in GTEx UKB GTEx_1000G UKB_1000G; do     # drop the ones already run
  case ${dataset} in
    GTEx)       res_dir="${base_dir}/Gtex_colocalization/res" ;;
    UKB)        res_dir="${base_dir}/UKB_coloclization/result" ;;
    GTEx_1000G) res_dir="${base_dir}/Gtex_colocalization_1000G/res" ;;
    UKB_1000G)  res_dir="${base_dir}/UKB_coloclization_1000_genome/result" ;;
  esac
  sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|DATASET|${dataset}|g" -e "s|RES_DIR|${res_dir}|g" \
      ${job_name}.sh > ${job_name}_${dataset}.sh.submit
  sb ${job_name}_${dataset}.sh.submit
done

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="Fig5_PR_ROC"
job_dir="${base_dir}/${job_name}"
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_DATASET
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=70000]
#BSUB -W 48:00
#BSUB -o WORK_DIR/log/JOB_NAME_DATASET.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_DATASET.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/35.Fig5_PR_ROC_Extract.ipynb" pr_extract \
    --dataset DATASET --res_dir RES_DIR --units_file WORK_DIR/units/DATASET_pr_units.tsv \
    --cwd WORK_DIR/result --mem 60G -J 4
EOF

for dataset in GTEx_1000G; do     # drop the ones already run
  case ${dataset} in
    GTEx)       res_dir="${base_dir}/Gtex_colocalization/res" ;;
    UKB)        res_dir="${base_dir}/UKB_coloclization/result" ;;
    GTEx_1000G) res_dir="${base_dir}/Gtex_colocalization_1000G/res" ;;
    UKB_1000G)  res_dir="${base_dir}/UKB_coloclization_1000_genome/result" ;;
  esac
  sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|DATASET|${dataset}|g" -e "s|RES_DIR|${res_dir}|g" \
      ${job_name}.sh > ${job_name}_${dataset}.sh.submit
  sb ${job_name}_${dataset}.sh.submit
done

## 3. Check

In [1]:
library(tidyverse)
root <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Fig5_PR_ROC"
for (ds in c("GTEx", "UKB", "GTEx_1000G", "UKB_1000G")) {
  if (!file.exists(file.path(root, "units", paste0(ds, "_pr_units.tsv")))) next
  units <- read_tsv(file.path(root, "units", paste0(ds, "_pr_units.tsv")), show_col_types = FALSE)
  ch    <- list.files(file.path(root, "result"), pattern = paste0("^", ds, "_pr_chunk_[0-9]+\\.rds$"), full.names = TRUE)
  u     <- map_dfr(ch, ~ readRDS(.x)$unit)
  cat(sprintf("\n== %s: %d of %d chunks, %d of %d units\n", ds, length(ch), n_distinct(units$chunk), nrow(u), nrow(units)))
  print(count(u, status, eligible))
  u %>% filter(status == "ok", eligible) %>%
    summarise(units = n(), genes_or_regions = n_distinct(gene), pct_ser_fallback = 100 * mean(ser_fallback),
              with_insample_cs = sum(n_cs_truth > 0), median_p = median(p)) %>% print()
}

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors



== GTEx: 40 of 40 chunks, 2600 of 2600 units
# A tibble: 2 × 3
  status eligible     n
  <chr>  <lgl>    <int>
1 ok     FALSE        1
2 ok     TRUE      2599
# A tibble: 1 × 5
  units genes_or_regions pct_ser_fallback with_insample_cs median_p
  <int>            <int>            <dbl>            <int>    <int>
1  2599             1570             1.15             2562     3301

== UKB: 40 of 40 chunks, 2600 of 2600 units
# A tibble: 2 × 3
  status eligible     n
  <chr>  <lgl>    <int>
1 ok     FALSE        4
2 ok     TRUE      2596
# A tibble: 1 × 5
  units genes_or_regions pct_ser_fallback with_insample_cs median_p
  <int>            <int>            <dbl>            <int>    <dbl>
1  2596              836             3.51             2549    3696.

== GTEx_1000G: 40 of 40 chunks, 2600 of 2600 units
# A tibble: 2 × 3
  status eligible     n
  <chr>  <lgl>    <int>
1 ok     FALSE        1
2 ok     TRUE      2599
# A tibble: 1 × 5
  units genes_or_regions pct_ser_fallback with_insamp

## 4. Submit jobs to compute the curves

Runs `Data_Processing/Figure_5/Fig5_Power_FDR.ipynb` (all its code cells, as an R script) once per reference LD, after section 2 has finished: 2 datasets × 3 in-sample PIP thresholds × 2 methods × 1000 bootstrap resamples, forked over 6 cores (minutes). Writes `figure_data/Fig5_Power_FDR_*` (ADSP) and `figure_data/Fig5_Power_FDR_1000G_*` (1000 Genomes) in the repo; the log (`log/Fig5_Power_FDR_<ld_ref>.*.out`) has the power at FDR 0.05 / 0.10 / 0.20 per curve and the summary tables. The notebook can also run interactively.

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="Fig5_Power_FDR"
job_dir="${base_dir}/Fig5_PR_ROC"
mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_LDREF
#BSUB -n 6
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 4:00
#BSUB -o WORK_DIR/log/JOB_NAME_LDREF.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME_LDREF.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
export LD_REF=LDREF
repo="/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper"
script="WORK_DIR/code/JOB_NAME_LDREF.R"
python3 -c "import json, sys; nb = json.load(open(sys.argv[1])); open(sys.argv[2], 'w').write('\n\n'.join(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code'))" \
    "${repo}/Data_Processing/Figure_5/Fig5_Power_FDR.ipynb" "${script}"
cd "${repo}/Data_Processing/Figure_5"
Rscript -e "source('${script}', print.eval = TRUE)"
EOF

for ld_ref in ADSP 1000G; do     # drop the one not needed
  sed -e "s|WORK_DIR|${job_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|LDREF|${ld_ref}|g" \
      ${job_name}.sh > ${job_name}_${ld_ref}.sh.submit
  sb ${job_name}_${ld_ref}.sh.submit
done